# 01 — Score a tournament from scratch

The full pipeline on the 2022 FIFA World Cup (StatsBomb open data): load events, compute every measure, standardize against the group stage, and publish the 0-10 board.

The weights were selected on 2026 data only. On the 2022 data, the index rates the Argentina-France final 9.59 and ranks it first of 64 matches.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))
os.environ.setdefault("EXCITEMENT_INDEX_CACHE", os.path.abspath("../.opendata_cache"))
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 140)

## Load matches and build the feature matrix

About 57 measures per match. The first run downloads and caches event data (roughly 5 minutes); subsequent runs read the cache.

In [2]:
from pathlib import Path
from excitement_index import opendata, build_feature_matrix

CACHE = Path("wc2022_features.parquet")
matches = opendata.load_matches("FIFA World Cup", "2022")
if CACHE.exists():
    features = pd.read_parquet(CACHE)
else:
    features = build_feature_matrix(matches, opendata.load_events,
                                    elo=opendata.load_elo(), jeopardy=False)
    features.to_parquet(CACHE)
print(features.shape)

(64, 64)


## Score

The reference pool is the tournament's group stage: every measure is standardized against those 48 matches.

In [3]:
from excitement_index import score_matches

group_ids = features.index[features["knockout"] == 0]
board = score_matches(features, reference_ids=list(group_ids))
board["rank"] = range(1, len(board) + 1)
board[["rank", "home", "away", "stage", "rating"]].head(15)

,rank,home,away,stage,rating
match_id,,,,,
3869685,1,Argentina,France,final,9.59
3869420,2,Croatia,Brazil,quarter-finals,9.49
3857292,3,Costa Rica,Germany,group,9.46
3869321,4,Netherlands,Argentina,quarter-finals,9.37
3869354,5,England,France,quarter-finals,9.32
3857284,6,Germany,Japan,group,9.26
3857259,7,Cameroon,Serbia,group,9.08
3869219,8,Japan,Croatia,round of 16,9.06
3857256,9,Serbia,Switzerland,group,8.84


## The final's decomposition

The five bucket values sum to the raw score.

In [4]:
final = board.loc[3869685]   # Argentina vs France
print(f"Argentina 3-3 France (4-2 pens)  ->  {final['rating']:.2f}/10, rank #{int(final['rank'])} of {len(board)}")
final[[c for c in board.columns if c.startswith('bucket_') or c.startswith('tax_')]]

Argentina 3-3 France (4-2 pens)  ->  9.59/10, rank #1 of 64


bucket_stakes       0.404816
bucket_chances      0.214174
bucket_drama        0.210654
bucket_spectacle    0.184956
bucket_payoff       0.324376
tax_dead_rubber         -0.0
tax_aliveness           -0.0
Name: 3869685, dtype: object

## The bottom of the board

In [5]:
board[["rank", "home", "away", "rating"]].tail(5)

,rank,home,away,rating
match_id,,,,
3857301,60,Qatar,Senegal,6.14
3857289,61,Argentina,Mexico,5.91
3857287,62,Uruguay,South Korea,5.74
3857294,63,Netherlands,Qatar,5.32
3857286,64,Qatar,Ecuador,5.07
